Guilherme Da Silva Ferreira Batista

In [1]:
"""
orcamento_empresa.py
====================
Sistema de Cálculo de Orçamento Corporativo com Auditoria.

Simula a estrutura hierárquica de uma multinacional e calcula
o orçamento total com suporte a departamentos ignorados,
conversão de moeda e registro de auditoria via decorator.
"""

import time


# =============================================================================
# 1. ESTRUTURA DE DADOS — Dicionário aninhado simulando a empresa
# =============================================================================

empresa = {
    "Matriz": {
        "TI": {
            "Infraestrutura": {
                "Servidores": 150_000,
                "Redes": 80_000,
                "Seguranca": 60_000,
            },
            "Desenvolvimento": {
                "Backend": 200_000,
                "Frontend": 180_000,
                "QA": 90_000,
            },
        },
        "RH": {
            "Recrutamento": {
                "Headhunting": 45_000,
                "Triagem": 20_000,
            },
            "Treinamento": {
                "Capacitacao": 35_000,
                "Onboarding": 15_000,
            },
            "Beneficios": 70_000,
        },
        "Financeiro": {
            "Contabilidade": 110_000,
            "Auditoria_Interna": 95_000,
            "Tesouraria": 130_000,
        },
        "Marketing": {
            "MidiasPagas": 250_000,
            "Conteudo": 80_000,
            "Branding": 60_000,
        },
    },
    "Filial_SP": {
        "Operacoes": {
            "Logistica": 175_000,
            "Estoque": 90_000,
        },
        "Vendas": {
            "Varejo": 300_000,
            "Atacado": 220_000,
            "Ecommerce": 410_000,
        },
        "Suporte": 55_000,
    },
    "Filial_RJ": {
        "Operacoes": {
            "Logistica": 140_000,
            "Estoque": 70_000,
        },
        "Vendas": {
            "Varejo": 190_000,
            "Atacado": 160_000,
        },
        "Juridico": {
            "Contratos": 85_000,
            "Compliance": 75_000,
        },
    },
}


# =============================================================================
# 2. DECORATOR — @auditor
#    Intercepta a chamada, imprime cabeçalho, registra args/kwargs
#    e mede o tempo total de execução.
# =============================================================================

def auditor(func):
    """
    Decorator de auditoria.
    Envolve qualquer função e imprime:
      - Cabeçalho indicando início do cálculo
      - Argumentos posicionais e nomeados recebidos
      - Tempo total de execução
      - Rodapé confirmando o encerramento
    """

    def wrapper(*args, **kwargs):  # repassa todos os args/kwargs à função original
        # --- Cabeçalho de auditoria ---
        print("=" * 60)
        print("          SISTEMA DE AUDITORIA — INÍCIO DO CÁLCULO")
        print("=" * 60)
        print(f"  Função auditada  : {func.__name__}")
        print(f"  Departamentos ignorados (*args) : {args[1:] or 'Nenhum'}")
        print(f"  Parâmetros de conversão (**kwargs): {kwargs or 'Nenhum'}")
        print("-" * 60)

        # --- Medição do tempo de execução ---
        inicio = time.perf_counter()
        resultado = func(*args, **kwargs)       # chama a função real
        fim = time.perf_counter()

        duracao = fim - inicio

        # --- Rodapé de auditoria ---
        print("-" * 60)
        print(f"  Tempo de processamento : {duracao:.6f} segundos")
        print("=" * 60)
        print("          SISTEMA DE AUDITORIA — CÁLCULO ENCERRADO")
        print("=" * 60)

        return resultado

    return wrapper


# =============================================================================
# 3. FUNÇÃO PRINCIPAL — calcular_orcamento
#    Navega recursivamente pela árvore de dicionários.
#    *args  → nomes de departamentos a ignorar
#    **kwargs → moeda_destino e taxa_cambio para conversão
# =============================================================================

@auditor
def calcular_orcamento(estrutura, *args, **kwargs):
    """
    Calcula recursivamente o orçamento total de uma estrutura
    hierárquica (dicionário aninhado).

    Parâmetros
    ----------
    estrutura : dict
        Dicionário (possivelmente aninhado) com os orçamentos.
    *args : str
        Nomes de departamentos a serem ignorados na soma.
        Todos os seus sub-departamentos também são excluídos.
    **kwargs
        moeda_destino (str) : símbolo da moeda de saída (ex: "USD").
        taxa_cambio   (float): fator de conversão a aplicar no total.

    Retorna
    -------
    float
        Soma total dos orçamentos, convertida pela taxa_cambio (se fornecida).
    """

    # Lê os parâmetros de conversão (valores padrão caso não sejam passados)
    moeda_destino = kwargs.get("moeda_destino", "BRL")
    taxa_cambio   = kwargs.get("taxa_cambio", 1.0)

    # Função auxiliar interna que faz a recursão real
    # (separada para que a taxa_cambio só seja aplicada UMA vez, ao final)
    def _somar(no, ignorados):
        """
        Percorre 'no' recursivamente e acumula os valores numéricos,
        pulando qualquer chave presente em 'ignorados'.
        """
        total = 0

        for chave, valor in no.items():

            # Verifica se o departamento atual deve ser ignorado
            if chave in ignorados:
                print(f"  [IGNORADO] Departamento '{chave}' excluído da soma.")
                continue  # pula este nó e todos os seus filhos

            if isinstance(valor, dict):
                # Nó intermediário → desce um nível na recursão
                total += _somar(valor, ignorados)

            elif isinstance(valor, (int, float)):
                # Nó folha → valor numérico encontrado, acumula
                total += valor

            else:
                # Tipo inesperado → avisa mas não interrompe o processamento
                print(f"  [AVISO] Valor ignorado em '{chave}': tipo {type(valor).__name__}")

        return total

    # Executa a recursão com os departamentos ignorados vindos de *args
    total_brl = _somar(estrutura, set(args))

    # Aplica a conversão de moeda
    total_convertido = total_brl * taxa_cambio

    # Exibe o resumo financeiro
    print(f"\n  Orçamento bruto (BRL)     : R$ {total_brl:>15,.2f}")
    if taxa_cambio != 1.0:
        print(f"  Taxa de câmbio aplicada   :    {taxa_cambio:>15.4f}")
        print(f"  Orçamento convertido ({moeda_destino:>3}): {moeda_destino} {total_convertido:>14,.2f}")
    print()

    return total_convertido


# =============================================================================
# 4. EXECUÇÃO — três cenários de uso para demonstrar os recursos
# =============================================================================

if __name__ == "__main__":

    # --- Cenário 1: Orçamento total sem filtros, sem conversão ---
    print("\n>>> CENÁRIO 1: Orçamento total (sem filtros, sem conversão)\n")
    total1 = calcular_orcamento(empresa)
    print(f"  Resultado final: R$ {total1:,.2f}\n")

    # --- Cenário 2: Ignorando departamentos de Marketing e Juridico ---
    print("\n>>> CENÁRIO 2: Ignorando Marketing e Juridico\n")
    total2 = calcular_orcamento(empresa, "Marketing", "Juridico")
    print(f"  Resultado final: R$ {total2:,.2f}\n")

    # --- Cenário 3: Convertendo para USD, ignorando Filial_RJ inteira ---
    print("\n>>> CENÁRIO 3: Convertendo para USD (taxa 0.19), ignorando Filial_RJ\n")
    total3 = calcular_orcamento(
        empresa,
        "Filial_RJ",
        moeda_destino="USD",
        taxa_cambio=0.19,
    )
    print(f"  Resultado final: USD {total3:,.2f}\n")


>>> CENÁRIO 1: Orçamento total (sem filtros, sem conversão)

          SISTEMA DE AUDITORIA — INÍCIO DO CÁLCULO
  Função auditada  : calcular_orcamento
  Departamentos ignorados (*args) : Nenhum
  Parâmetros de conversão (**kwargs): Nenhum
------------------------------------------------------------

  Orçamento bruto (BRL)     : R$    3,640,000.00

------------------------------------------------------------
  Tempo de processamento : 0.000033 segundos
          SISTEMA DE AUDITORIA — CÁLCULO ENCERRADO
  Resultado final: R$ 3,640,000.00


>>> CENÁRIO 2: Ignorando Marketing e Juridico

          SISTEMA DE AUDITORIA — INÍCIO DO CÁLCULO
  Função auditada  : calcular_orcamento
  Departamentos ignorados (*args) : ('Marketing', 'Juridico')
  Parâmetros de conversão (**kwargs): Nenhum
------------------------------------------------------------
  [IGNORADO] Departamento 'Marketing' excluído da soma.
  [IGNORADO] Departamento 'Juridico' excluído da soma.

  Orçamento bruto (BRL)     : R$   